# Edge check: guava (Java)
Expected edges come from **javac's own parser** (`java-ref/ListEdges.java`): parse only, no classpath, nothing compiled, not tree-sitter.
They are compared with code-review-graph (CRG) edges from `graph.db`, which is opened read-only.

| Edge | Reference (javac) | Match key |
|---|---|---|
| CONTAINS | class / method declarations and their parent | file + child start line + child name |
| INHERITS | `extends` / `implements` clauses | file + class start line + parent simple name |
| IMPORTS_FROM | `import` lines | file + line |
| CALLS | call sites `a.b()` and `new X()` | file + line + name (line of the name, else line where the call starts) |

- **Recall** = expected edges that CRG has. **Precision** = CRG edges that javac also has.
- CALLS here is call **sites by name**. It doesn't check *which* `b` is called, because that needs a full compile.
- Every run is saved to `results/guava/checks/<run>/` and logged in `results/guava/checks.csv`.

In [1]:
import json, sqlite3, subprocess, collections, pathlib, re, datetime
import pandas as pd
pd.set_option('display.max_colwidth', 90)
ROOT = pathlib.Path('..').resolve()
APP = 'guava'
REPO = ROOT / 'apps' / APP
CRG_RUN = 'iter-003_2026-11-17_2247'
DB = ROOT / f'results/{APP}/runs/{CRG_RUN}/crg/graph.db'
RUN = 'edges_' + datetime.datetime.now().strftime('%Y-%m-%d_%H%M')
OUT = ROOT / f'results/{APP}/checks/{RUN}'; OUT.mkdir(parents=True, exist_ok=True)
REF_COMMIT = subprocess.run(['git','rev-parse','--short','HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip()
print(RUN, '| repo commit', REF_COMMIT, '| CRG run', CRG_RUN)

edges_2026-11-20_1158 | repo commit fce80fd | CRG run iter-003_2026-11-17_2247


## 1. Reference: javac

In [2]:
files = subprocess.run(['git','ls-files','*.java'], cwd=REPO, capture_output=True, text=True, check=True).stdout
out = subprocess.run(['java', str(ROOT/'java-ref/ListEdges.java'), str(REPO)], input=files, capture_output=True, text=True, check=True).stdout
rows = [json.loads(l) for l in out.splitlines()]
(OUT/'ref_edges.jsonl').write_text(out)
R = {e: pd.DataFrame([r for r in rows if r['e'] == e]) for e in ['contains','inherits','import','call']}
print('java files:', len(files.split()), '| errors:', sum(r['e']=='error' for r in rows))
pd.Series({e: len(d) for e, d in R.items()}, name='javac rows')

java files: 3267 | errors: 0


contains     71080
inherits      6231
import       39233
call        301434
Name: javac rows, dtype: int64

## 2. code-review-graph edges

In [3]:
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
P = str(REPO) + '/'
rel = lambda s: s.str.replace(P, '', regex=False)
nodes = pd.read_sql('select qualified_name, name, line_start, kind from nodes', con); nodes['qualified_name'] = rel(nodes.qualified_name)
def crg(kind):
    d = pd.read_sql("select source_qualified src, target_qualified tgt, file_path file, line from edges where kind=? and file_path like '%.java'", con, params=(kind,))
    for c in ['src','tgt','file']: d[c] = rel(d[c])
    return d
G = {k: crg(k) for k in ['CONTAINS','INHERITS','IMPORTS_FROM','CALLS']}
pd.Series({k: len(d) for k, d in G.items()}, name='CRG rows (.java files)')

CONTAINS         66448
INHERITS          6001
IMPORTS_FROM     39233
CALLS           287769
Name: CRG rows (.java files), dtype: int64

Matching: each edge is used at most once (multiset). A second key is tried only for rows still unmatched.

In [4]:
def match(ref, crg, passes):
    ref = ref.copy(); crg = crg.copy(); ref['hit'] = False; crg['hit'] = False
    for rk, ck in passes:
        rr = ref[~ref.hit].copy(); cc = crg[~crg.hit].copy()
        rr['_n'] = rr.groupby(rk).cumcount(); cc['_n'] = cc.groupby(ck).cumcount()
        j = rr.reset_index().merge(cc.reset_index(), left_on=rk+['_n'], right_on=ck+['_n'], suffixes=('','_c'))
        ref.loc[j['index'], 'hit'] = True; crg.loc[j['index_c'], 'hit'] = True
    return ref, crg
SUMMARY = []
def report(edge, ref, crg):
    m, e = int(ref.hit.sum()), int(crg.hit.sum())
    SUMMARY.append({'edge': edge, 'expected (javac)': len(ref), 'CRG': len(crg), 'matched': m,
                    'missed': len(ref)-m, 'CRG extra': len(crg)-e,
                    'recall %': round(100*m/len(ref), 1), 'precision %': round(100*e/len(crg), 1)})
    ref[~ref.hit].drop(columns='hit').to_csv(OUT/f'{edge}_missed.csv', index=False)
    crg[~crg.hit].drop(columns='hit').to_csv(OUT/f'{edge}_extra.csv', index=False)
    return pd.DataFrame([SUMMARY[-1]]).set_index('edge')

## 3. CONTAINS

In [5]:
c = G['CONTAINS'].merge(nodes, left_on='tgt', right_on='qualified_name', how='left')
c = c.rename(columns={'line_start':'child_line','name':'child'})
rc, cc = match(R['contains'], c, [(['file','line','child'], ['file','child_line','child'])])
report('CONTAINS', rc, cc)

,expected (javac),CRG,matched,missed,CRG extra,recall %,precision %
edge,,,,,,,
CONTAINS,71080,66448,66448,4632,0,93.5,100.0


In [6]:
miss = rc[~rc.hit]
miss.assign(why=miss.child.eq('').map({True:'anonymous class'}).fillna(
         miss.parent.eq('').map({True:'method of anonymous class'})).fillna(miss.kind)).why.value_counts()

why
anonymous class              4444
method of anonymous class     107
method                         48
class                          33
Name: count, dtype: int64

## 4. INHERITS

In [7]:
def simple(t):
    t = re.split(r'[<\s]', t.strip(), maxsplit=1)[0]      # drop generics and anything after whitespace
    return t.split('.')[-1]
i = G['INHERITS'].merge(nodes, left_on='src', right_on='qualified_name', how='left')
i['sup_simple'] = i.tgt.map(simple); i['raw_ok'] = ~i.tgt.str.contains('\n')
ri = R['inherits'].assign(sup_simple=R['inherits'].sup.map(lambda s: s.split('.')[-1]))
ri, ci = match(ri, i, [(['file','line','sup_simple'], ['file','line_start','sup_simple'])])
display(report('INHERITS', ri, ci))
print('CRG targets with broken text (newline inside the name):', int((~i.raw_ok).sum()))

,expected (javac),CRG,matched,missed,CRG extra,recall %,precision %
edge,,,,,,,
INHERITS,6231,6001,6001,230,0,96.3,100.0


CRG targets with broken text (newline inside the name): 26


In [8]:
ri[~ri.hit][['file','line','cls','sup','how']].head(10)

,file,line,cls,sup,how
52,android/guava-testlib/src/com/google/common/collect/testing/OneSizeTestContainerGenera...,35,OneSizeTestContainerGenerator,TestSubjectGenerator,extends
53,android/guava-testlib/src/com/google/common/collect/testing/OneSizeTestContainerGenera...,35,OneSizeTestContainerGenerator,TestContainerGenerator,extends
80,android/guava-testlib/src/com/google/common/collect/testing/TestCollectionGenerator.java,29,TestCollectionGenerator,TestContainerGenerator,extends
86,android/guava-testlib/src/com/google/common/collect/testing/TestListGenerator.java,29,TestListGenerator,TestCollectionGenerator,extends
88,android/guava-testlib/src/com/google/common/collect/testing/TestMapGenerator.java,29,TestMapGenerator,TestContainerGenerator,extends
89,android/guava-testlib/src/com/google/common/collect/testing/TestQueueGenerator.java,29,TestQueueGenerator,TestCollectionGenerator,extends
90,android/guava-testlib/src/com/google/common/collect/testing/TestSetGenerator.java,29,TestSetGenerator,TestCollectionGenerator,extends
91,android/guava-testlib/src/com/google/common/collect/testing/TestSortedMapGenerator.java,30,TestSortedMapGenerator,TestMapGenerator,extends
92,android/guava-testlib/src/com/google/common/collect/testing/TestSortedSetGenerator.java,29,TestSortedSetGenerator,TestSetGenerator,extends
273,android/guava-testlib/src/com/google/common/collect/testing/google/TestBiMapGenerator....,31,TestBiMapGenerator,TestContainerGenerator,extends


## 5. IMPORTS_FROM

In [9]:
ri2, ci2 = match(R['import'], G['IMPORTS_FROM'], [(['file','line'], ['file','line'])])
display(report('IMPORTS_FROM', ri2, ci2))
t = ci2.tgt
print('CRG import target is a repo file:', int(t.str.endswith('.java').sum()), '| other (name as written):', int((~t.str.endswith('.java')).sum()))
ri2[~ri2.hit].assign(kind=lambda d: d.static.map({True:'static'}).fillna('')+d.star.map({True:' star'}).fillna('')).kind.value_counts()

,expected (javac),CRG,matched,missed,CRG extra,recall %,precision %
edge,,,,,,,
IMPORTS_FROM,39233,39233,39233,0,0,100.0,100.0


CRG import target is a repo file: 8484 | other (name as written): 30749


Series([], Name: count, dtype: int64)

## 6. CALLS (call sites by name)

In [10]:
k = G['CALLS'].copy()
k['name'] = k.tgt.map(lambda t: t.split('::')[-1].split('.')[-1].split(':')[0])   # 'Cls.m:S<hash>' -> 'm'
rk, ck = match(R['call'], k, [(['file','line','name'], ['file','line','name']),
                              (['file','start','name'], ['file','line','name'])])
report('CALLS', rk, ck)

,expected (javac),CRG,matched,missed,CRG extra,recall %,precision %
edge,,,,,,,
CALLS,301434,287769,286727,14707,1042,95.1,99.6


In [11]:
miss = rk[~rk.hit]
miss.assign(why=miss.name.isin(['super','this']).map({True:'super(...) / this(...)'}).fillna('')
            .where(lambda s: s != '', miss.how.map({'new':'new X()','call':'a.b() / b()'}))).why.value_counts()

why
new X()                   6749
a.b() / b()               6681
super(...) / this(...)    1277
Name: count, dtype: int64

In [12]:
ck[~ck.hit][['file','line','src','tgt']].head(10)

,file,line,src,tgt
922,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,289,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear
977,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,742,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear
1029,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,480,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove
1030,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,483,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove
1254,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1249,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear
1471,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1514,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear
1484,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1342,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove
1485,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1345,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove
2288,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java,312,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java::Sp...,add
2309,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java,288,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java::Sp...,accept


## 7. Tree-sitter module (`core-module/edges.py`)
Same javac reference and the same match keys as for CRG. Rows are logged as `<EDGE> (module)`.

Rules (each one comes from what javac reports):
- Annotation types (`@interface`) and anonymous classes are classes. An anonymous class sits on the line of its `{`.
- An enum constant is a call `new Enum(...)`, on the line of its name.
- `super(...)` / `this(...)` are calls named `super` / `this`.

In [13]:
import sys; sys.path.insert(0, str(ROOT/'core-module'))
import importlib, edges as module_edges_mod; importlib.reload(module_edges_mod)
M = {k: [] for k in ['contains','inherits','import','call']}
for f in files.split():
    for k, v in module_edges_mod.module_edges(REPO, f).items(): M[k] += v
M = {k: pd.DataFrame(v) for k, v in M.items()}
with open(OUT/'module_edges.jsonl', 'w') as fh:
    for k, d in M.items():
        for r in d.to_dict('records'): fh.write(json.dumps({'e': k, **r}) + '\n')
pd.Series({k: len(d) for k, d in M.items()}, name='module rows')

contains     71080
inherits      6231
import       39233
call        301434
Name: module rows, dtype: int64

In [14]:
mc_r, mc_m = match(R['contains'], M['contains'], [(['file','line','child'], ['file','line','child'])])
report('CONTAINS (module)', mc_r, mc_m)
mi_r, mi_m = match(R['inherits'].assign(sup_simple=R['inherits'].sup.map(lambda s: s.split('.')[-1])),
                   M['inherits'].rename(columns={'sup':'sup_simple'}), [(['file','line','sup_simple'], ['file','line','sup_simple'])])
report('INHERITS (module)', mi_r, mi_m)
mp_r, mp_m = match(R['import'], M['import'], [(['file','line'], ['file','line'])])
report('IMPORTS_FROM (module)', mp_r, mp_m)
mk_r, mk_m = match(R['call'], M['call'], [(['file','line','name'], ['file','line','name']), (['file','start','name'], ['file','start','name'])])
report('CALLS (module)', mk_r, mk_m)
S = pd.DataFrame(SUMMARY).set_index('edge')
cmp = pd.DataFrame({e: {'expected': S.loc[e,'expected (javac)'],
                        'CRG recall %': S.loc[e,'recall %'], 'module recall %': S.loc[f'{e} (module)','recall %'],
                        'CRG precision %': S.loc[e,'precision %'], 'module precision %': S.loc[f'{e} (module)','precision %'],
                        'CRG missed': S.loc[e,'missed'], 'module missed': S.loc[f'{e} (module)','missed']}
                    for e in ['CONTAINS','INHERITS','IMPORTS_FROM','CALLS']}).T
cmp

,expected,CRG recall %,module recall %,CRG precision %,module precision %,CRG missed,module missed
CONTAINS,71080.0,93.5,100.0,100.0,100.0,4632.0,0.0
INHERITS,6231.0,96.3,100.0,100.0,100.0,230.0,0.0
IMPORTS_FROM,39233.0,100.0,100.0,100.0,100.0,0.0,0.0
CALLS,301434.0,95.1,100.0,99.6,100.0,14707.0,0.0


In [15]:
# anything the module still misses or adds
pd.concat([d[~d.hit].assign(edge=e, side=s) for e, s, d in [('CONTAINS','missed',mc_r),('CONTAINS','extra',mc_m),
           ('INHERITS','missed',mi_r),('INHERITS','extra',mi_m),('IMPORTS','missed',mp_r),('IMPORTS','extra',mp_m),
           ('CALLS','missed',mk_r),('CALLS','extra',mk_m)]]).groupby(['edge','side']).size()

Series([], dtype: int64)

## 8. Summary and save

In [16]:
S = pd.DataFrame(SUMMARY).set_index('edge'); S.to_csv(OUT/'summary.csv')
log = ROOT / f'results/{APP}/checks.csv'
S.reset_index().assign(run=RUN, repo_commit=REF_COMMIT, crg_run=CRG_RUN, reference='javac parse (java-ref/ListEdges.java)') \
 .to_csv(log, mode='a', header=not log.exists(), index=False)
print('saved to', OUT.relative_to(ROOT), '| logged in', log.relative_to(ROOT))
S

saved to results/guava/checks/edges_2026-11-20_1158 | logged in results/guava/checks.csv


,expected (javac),CRG,matched,missed,CRG extra,recall %,precision %
edge,,,,,,,
CONTAINS,71080,66448,66448,4632,0,93.5,100.0
INHERITS,6231,6001,6001,230,0,96.3,100.0
IMPORTS_FROM,39233,39233,39233,0,0,100.0,100.0
CALLS,301434,287769,286727,14707,1042,95.1,99.6
CONTAINS (module),71080,71080,71080,0,0,100.0,100.0
INHERITS (module),6231,6231,6231,0,0,100.0,100.0
IMPORTS_FROM (module),39233,39233,39233,0,0,100.0,100.0
CALLS (module),301434,301434,301434,0,0,100.0,100.0
